# nanoserve — GPU headline run

A from-scratch LLM inference server. This notebook runs the full fp16 pipeline on a CUDA GPU and produces the headline graphs + numbers.

**To run:** `Runtime → Change runtime type → T4 GPU` (usually preselected), then `Runtime → Run all`. ~20 minutes. That's it.

In [ ]:
# 1. confirm we actually have a GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. get the code (fresh clone each run so re-runs pick up fixes)
!rm -rf /content/nanoserve && git clone -q https://github.com/Savi-Swar/nanoserve.git /content/nanoserve
%cd /content/nanoserve

In [ ]:
# 3. install deps (vLLM is optional — it provides the reference ceiling)
!pip install -q -r requirements.txt
!pip install -q vllm || echo 'vLLM install skipped/failed — the ceiling step will be skipped, everything else runs'

In [ ]:
# 4. fetch the real Azure LLM inference trace
!curl -sL -o data/azure_llm_conv.csv https://raw.githubusercontent.com/Azure/AzurePublicDataset/master/data/AzureLLMInferenceTrace_conv.csv
!wc -l data/azure_llm_conv.csv

In [ ]:
# 5. the whole run: ladder + trace + audit + noise floor + vLLM ceiling + roofline
!python scripts/gpu_run.py

In [ ]:
# 6. show the headline graphs
from IPython.display import Image, display
for f in ['throughput_vs_rate', 'ttft_p99_by_engine', 'throughput_by_engine', 'latency_throughput']:
    try:
        display(Image(f'results/{f}.png'))
    except Exception as e:
        print(f'{f}: {e}')

In [ ]:
# 7. headline numbers (also saved to results/summary.txt), then optional zip download
!cat results/summary.txt
try:
    !zip -qr nanoserve_results.zip results
    from google.colab import files
    files.download('nanoserve_results.zip')
except Exception as e:
    print('zip/download skipped:', e)